# 02. 고영향 인공지능 판단 가이드라인 전처리

**수정 사항**: 줄바꿈으로 쪼개진 한글 단어 복원 + 섹션 분리 안정성 개선

In [1]:
#!pip install pdfplumber -q

In [ ]:
import pdfplumber, re, json
from pathlib import Path

PDF_PATH = "/Users/gimseoyeon/Downloads/AICPP/data/processed/260429_고영향_인공지능_판단_가이드라인.pdf"
OUTPUT_PATH = "/Users/gimseoyeon/Downloads/AICPP/data/processed/판단가이드라인_chunks.jsonl"
CHUNK_SIZE, OVERLAP = 600, 100

assert Path(PDF_PATH).exists(), f"PDF 없음: {PDF_PATH}"
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
print("PDF 확인:", PDF_PATH)

## 1. 전체 텍스트 추출

In [3]:
def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            t = page.extract_text()
            if t and t.strip():
                pages.append({"page": i+1, "text": t.strip()})
    print(f"{len(pages)}개 페이지 추출")
    return pages

pages = extract_pages(PDF_PATH)


171개 페이지 추출


## 2. 텍스트 클리닝 (한글 단어 줄바꿈 복원 포함)

In [4]:
def clean(text):
    # 반복 헤더·푸터 제거
    text = re.sub(r'고영향 인공지능 판단 가이드라인\s*', '', text)
    text = re.sub(r'과학기술정보통신부\s*', '', text)
    # ★ 핵심 수정: 한글 단어가 줄바꿈으로 쪼개진 경우 이어붙이기
    text = re.sub(r'([가-힣])\n([가-힣])', r'\1\2', text)
    text = re.sub(r'([가-힣])\n([가-힣])', r'\1\2', text)  # 2회 적용
    # 영문/숫자 + 한글 연결
    text = re.sub(r'([a-zA-Z0-9])\n([가-힣])', r'\1 \2', text)
    # 단독 숫자 줄(페이지번호) 제거
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

for p in pages:
    p["clean"] = clean(p["text"])


## 3. 섹션 헤더 감지 및 청킹

In [5]:
# 섹션 패턴: 1. / 1.1 / 1.1.1 형태
SECTION_PAT = re.compile(
    r'^(\d+(?:\.\d+){0,2})\.?\s+(.+)$',
    re.MULTILINE
)

def has_legal_ref(text):
    return bool(re.search(r'제\d+조|제\d+항|별표|부칙', text))

def overlap_chunk(text, size=CHUNK_SIZE, overlap=OVERLAP):
    """텍스트를 size 글자씩, overlap 글자 겹쳐서 분리."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

full_text = "\n\n".join(p["clean"] for p in pages)

# 섹션별로 분리
sections = []
matches = list(SECTION_PAT.finditer(full_text))
for i, m in enumerate(matches):
    sec_id   = m.group(1)
    sec_title= m.group(2).strip()
    start    = m.end()
    end      = matches[i+1].start() if i+1 < len(matches) else len(full_text)
    body     = full_text[start:end].strip()
    if len(body) < 30:
        continue
    sections.append({"id": sec_id, "title": sec_title, "body": body})

print(f"섹션 {len(sections)}개 감지")
for s in sections[:5]:
    print(f"  [{s['id']}] {s['title'][:40]}")


섹션 229개 감지
  [1.1.1] 고영향 인공지능 가이드라인의 수립 배경
  [5] 그 밖에 고영향 인공지능의 해당 여부를 확인하는 데 필요한 자료로서 과학
  [5] 그 밖에 고영향 인공지능의 해당 여부를 확인하는 데 필요한 자료로서 장관
  [1.2.1] 인공지능기본법 제2조(정의)에 따른 주요 용어용어 개념
  [1.2.2] 인공지능시스템의 개념인공지능시스템은 다양한 수준의 2자율성과 3적응성을 


## 4. JSONL 저장

In [8]:
chunks = []
cid = 0
for sec in sections:
    sub_chunks = overlap_chunk(sec["body"])
    for j, sub in enumerate(sub_chunks):
        sub = re.sub(r'\s+', ' ', sub).strip()
        if not sub:
            continue
        chunk = {
            "chunk_id":    f"판단가이드_{cid:04d}",
            "section_id":  sec["id"],
            "title":       sec["title"],
            "content":     f"[{sec['id']} {sec['title']}]\n{sub}",
            "has_legal_ref": has_legal_ref(sub),
            "출처":        "고영향 인공지능 판단 가이드라인",
            "청크유형":    "섹션청크",
        }
        chunks.append(chunk)
        cid += 1

print(f"총 {len(chunks)}개 청크 생성")
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"저장 완료: {OUTPUT_PATH}")


총 443개 청크 생성
저장 완료: /Users/gimseoyeon/Downloads/AICPP/data/processed/판단가이드라인_chunks.jsonl


## 5. 검증

In [9]:
legal_chunks = [c for c in chunks if c["has_legal_ref"]]
print(f"법 조항 참조 청크: {len(legal_chunks)}개 / 전체 {len(chunks)}개")
print("\n샘플:")
for c in chunks[:2]:
    print(f"  [{c['section_id']}] {c['content'][:120]}...")


법 조항 참조 청크: 214개 / 전체 443개

샘플:
  [1.1.1] [1.1.1 고영향 인공지능 가이드라인의 수립 배경]
► 인공지능기본법 제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶...
  [1.1.1] [1.1.1 고영향 인공지능 가이드라인의 수립 배경]
의 기준과 예시 등을 마련하여 사업자의 예측 가능성을 제고하고 산업현장에서의혼란을 방지하기 위한 목적으로 마련되었다. Ÿ 본 가이드라인은 인공지능기본법이 추구하는...
